# Лабораторная работа №4
## Линейные модели, SVM и деревья решений.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б

### Цель работы
Изучение линейных моделей, метода опорных векторов (SVM) и деревьев решений.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Загрузка и разделение данных
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Обучение: {X_train_s.shape}, Тест: {X_test_s.shape}")


In [ ]:
# === 1. ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ ===
lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)

lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred, average='weighted')
print(f"Логистическая регрессия: accuracy={lr_acc:.4f}, F1={lr_f1:.4f}")

# Коэффициенты
print("\nКоэффициенты модели:")
coef_df = pd.DataFrame(lr.coef_, columns=iris.feature_names, index=iris.target_names)
print(coef_df.round(3))


In [ ]:
# === 2. SVM ===
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train_s, y_train)
svm_pred = svm.predict(X_test_s)

svm_acc = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')
print(f"SVM (RBF): accuracy={svm_acc:.4f}, F1={svm_f1:.4f}")

# Подбор ядра
for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVC(kernel=kernel, random_state=42)
    m.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_s))
    print(f"  SVM ({kernel}): accuracy={acc:.4f}")


In [ ]:
# === 3. ДЕРЕВО РЕШЕНИЙ ===
dt = DecisionTreeClassifier(max_depth=4, min_samples_split=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

dt_acc = accuracy_score(y_test, dt_pred)
dt_f1 = f1_score(y_test, dt_pred, average='weighted')
print(f"Дерево решений: accuracy={dt_acc:.4f}, F1={dt_f1:.4f}")

# Важность признаков
importance = pd.DataFrame({'feature': iris.feature_names, 'importance': dt.feature_importances_})
importance = importance.sort_values('importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'], color='steelblue')
plt.xlabel('Важность признака')
plt.title('Важность признаков в дереве решений', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab4/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(importance)


In [ ]:
# Визуализация дерева
plt.figure(figsize=(20, 12))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=10, impurity=False)
plt.title('Дерево решений для классификации Iris', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab4/decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === СРАВНЕНИЕ МОДЕЛЕЙ ===
results = pd.DataFrame({
    'Модель': ['Логистическая регрессия', 'SVM (RBF)', 'Дерево решений'],
    'Accuracy': [lr_acc, svm_acc, dt_acc],
    'F1-score': [lr_f1, svm_f1, dt_f1]
})
results = results.round(4)
print(results.to_string(index=False))

# Визуализация сравнения
x = np.arange(len(results))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, results['F1-score'], width, label='F1-score', color='coral')
ax.set_ylabel('Score')
ax.set_title('Сравнение моделей', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Модель'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0.8, 1.05)
ax.grid(True, alpha=0.3, axis='y')
# Значения на столбцах
for b in bars1+bars2:
    h = b.get_height()
    ax.annotate(f'{h:.3f}', xy=(b.get_x()+b.get_width()/2, h), xytext=(0,3),
                textcoords="offset points", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab4/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### Выводы
1. Все три модели показали высокое качество (accuracy >93%).
2. SVM (RBF) и логистическая регрессия — лучшие результаты.
3. Наиболее важный признак для дерева решений — petal length.
4. Дерево решений даёт интерпретируемые правила классификации.
5. Масштабирование важно для логистической регрессии и SVM.
